### Patient list Processing

In [3]:
import pandas as pd

# ---------------- File paths ----------------
file_mrn = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V3/2025_CXR/mrn.xlsx"
file_trap = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V3/2025_CXR/TRAP-PARDS_pullAug28.xlsx"

# ---------------- Helpers ----------------
def norm_pid_series(s):
    """Normalize PatientID series to comparable strings."""
    return s.astype(str).str.strip()

def get_pid_set(df):
    """Return a normalized set of PatientIDs from a DF if column exists; else empty set."""
    if "PatientID" not in df.columns:
        return set()
    return set(norm_pid_series(df["PatientID"].dropna()))

def report_inclusion(left_name, left_ids, right_name, right_ids, show_examples=15):
    """Check if all left_ids are included in right_ids; print summary + missing examples."""
    missing = left_ids - right_ids
    all_included = len(missing) == 0
    print(f"\nAll {left_name} PatientIDs included in {right_name}? {all_included}")
    print(f"  {left_name} unique IDs: {len(left_ids)}")
    print(f"  {right_name} unique IDs: {len(right_ids)}")
    if not all_included:
        print(f"  Missing in {right_name}: {len(missing)}")
        if len(missing) > 0:
            ex = list(missing)[:show_examples]
            print(f"  Examples (up to {show_examples}): {ex}")

# ---------------- Load MRN (single sheet) ----------------
df_mrn = pd.read_excel(file_mrn)

print("\nMRN.xlsx:")
print("Shape:", df_mrn.shape)
print(df_mrn.head())

# Duplicates in MRN
if "PatientID" in df_mrn.columns:
    dup_mrn = df_mrn[df_mrn.duplicated(subset=["PatientID"], keep=False)]
    print("\nDuplicate PatientIDs in MRN file:", dup_mrn["PatientID"].nunique())
    if not dup_mrn.empty:
        print(dup_mrn.sort_values("PatientID"))
else:
    print("\nWARNING: 'PatientID' column not found in MRN file.")

# ---------------- Load TRAP (all sheets) ----------------
trap_sheets = pd.read_excel(file_trap, sheet_name=None)
sheet_names = list(trap_sheets.keys())
print("\nSheets in TRAP-PARDS file:", sheet_names)

# Access sheets by order (Sheet2 = index 1, Sheet3 = index 2, Sheet4 = index 3)
try:
    df_trap1 = trap_sheets[sheet_names[0]]
    df_trap2 = trap_sheets[sheet_names[1]]
    df_trap3 = trap_sheets[sheet_names[2]]
    df_trap4 = trap_sheets[sheet_names[3]]
except IndexError:
    raise ValueError("Expected at least 4 sheets in the TRAP workbook.")

# Preview + duplicate checks for each sheet
for sheet_name, df in trap_sheets.items():
    print(f"\n===== Preview of sheet: {sheet_name} =====")
    print("Shape:", df.shape)
    print(df.head())

    if "PatientID" in df.columns:
        dup = df[df.duplicated(subset=["PatientID"], keep=False)]
        print(f"Duplicate PatientIDs in {sheet_name}: {dup['PatientID'].nunique()}")
        if not dup.empty:
            print(dup.sort_values("PatientID"))
    else:
        print(f"WARNING: 'PatientID' not found in {sheet_name}")

# ---------------- Normalize and build ID sets ----------------
mrn_ids   = get_pid_set(df_mrn)
sheet2_ids = get_pid_set(df_trap2)
sheet3_ids = get_pid_set(df_trap3)
sheet4_ids = get_pid_set(df_trap4)

# ---------------- Inclusion checks ----------------
# 1) All MRN PatientIDs are in Sheet2, Sheet3, and Sheet4
report_inclusion("MRN", mrn_ids, "Sheet2", sheet2_ids)
report_inclusion("MRN", mrn_ids, "Sheet3", sheet3_ids)
report_inclusion("MRN", mrn_ids, "Sheet4", sheet4_ids)

# 2) All Sheet2 PatientIDs are in Sheet3
report_inclusion("Sheet2", sheet2_ids, "Sheet3", sheet3_ids)

# 3) All Sheet3 PatientIDs are in Sheet4
report_inclusion("Sheet3", sheet3_ids, "Sheet4", sheet4_ids)



MRN.xlsx:
Shape: (899, 2)
                              PatientID        MRN
0  12288E12-375C-E311-800F-00215A9B0094   38759140
1  43C395B6-395C-E311-800F-00215A9B0094  100117962
2  D7970BC6-445C-E311-800F-00215A9B0094  100205702
3  1912AB08-455C-E311-800F-00215A9B0094  100151965
4  1E4D6298-455C-E311-800F-00215A9B0094   41994058

Duplicate PatientIDs in MRN file: 0

Sheets in TRAP-PARDS file: ['Query Specifications', 'EncounterAll', 'PatientInfo', 'FlowsheetsDetailed']

===== Preview of sheet: Query Specifications =====
Shape: (17, 3)
          Query Name: TRAP-PARDS_cxr - copy               Unnamed: 2
0  Query Description:                   NaN                      NaN
1                 NaN                   NaN                      NaN
2   Encounters Filter                   NaN                      NaN
3                 NaN           filter mode          Encounter Level
4                 NaN       date of service  01/01/2023 - 08/07/2025

===== Preview of sheet: EncounterAll =====

In [8]:
import pandas as pd

# ---- File paths ----
file_mrn  = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V3/2025_CXR/mrn.xlsx"
file_trap = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V3/2025_CXR/TRAP-PARDS_pullAug28.xlsx"

# ---- Load data ----
df_mrn = pd.read_excel(file_mrn)  # expected: PatientID, MRN
trap_sheets = pd.read_excel(file_trap, sheet_name=None)

# ---- Helpers ----
def norm_pid(s):
    return s.astype(str).str.strip()

def get_sheet_by_columns(sheets_dict, required_cols):
    for name, df in sheets_dict.items():
        if all(col in df.columns for col in required_cols):
            return name, df.copy()
    raise ValueError(f"No sheet found containing all columns: {required_cols}")

# Normalize PatientID in MRN + pad MRN to 9 digits
if "PatientID" not in df_mrn.columns:
    raise ValueError("MRN file must contain 'PatientID' column.")
df_mrn["PatientID"] = norm_pid(df_mrn["PatientID"])
if "MRN" in df_mrn.columns:
    df_mrn["MRN"] = df_mrn["MRN"].astype(str).str.zfill(9)

# Identify sheets by columns (robust to order)
# Sheet2-like: has InpatientAdmitDate; Sheet3-like: has DOB
name2, df2 = get_sheet_by_columns(trap_sheets, ["PatientID", "InpatientAdmitDate"])
name3, df3 = get_sheet_by_columns(trap_sheets, ["PatientID", "DOB"])

# Normalize PatientID
df2["PatientID"] = norm_pid(df2["PatientID"])
df3["PatientID"] = norm_pid(df3["PatientID"])

# Parse dates
df2["InpatientAdmitDate"] = pd.to_datetime(df2["InpatientAdmitDate"], errors="coerce")
df3["DOB"] = pd.to_datetime(df3["DOB"], errors="coerce")

# Filter to MRN-listed PatientIDs only (optional but cleaner)
mrn_ids = set(df_mrn["PatientID"])
df2_sub = df2[df2["PatientID"].isin(mrn_ids)].copy()
df3_sub = df3[df3["PatientID"].isin(mrn_ids)].copy()

# ===== Expand multiple inpatient admits into numbered columns =====
# Keep only valid dates, sort, enumerate per patient, then pivot wide.
df2_valid = df2_sub.dropna(subset=["InpatientAdmitDate"]).copy()
df2_valid = df2_valid.sort_values(["PatientID", "InpatientAdmitDate"])
df2_valid["admit_idx"] = df2_valid.groupby("PatientID").cumcount() + 1  # 1-based
wide_admits = (
    df2_valid
    .pivot(index="PatientID", columns="admit_idx", values="InpatientAdmitDate")
    .rename(columns=lambda i: f"InpatientAdmitDate_{int(i)}")
    .reset_index()
)

# ===== Prepare DOB (one per PatientID; keep earliest if multiple distinct) =====
dob_agg = (
    df3_sub.dropna(subset=["DOB"])
    .sort_values(["PatientID", "DOB"])
    .drop_duplicates(subset=["PatientID"], keep="first")
    [["PatientID", "DOB"]]
)

# Optional: warn about conflicting DOBs
dob_conflicts = (
    df3_sub.dropna(subset=["DOB"])
    .groupby("PatientID")["DOB"].nunique().reset_index(name="distinct_dob_count")
)
dob_conflicts = dob_conflicts[dob_conflicts["distinct_dob_count"] > 1]

# ===== Merge all together (keep all MRN rows) =====
final_df = (
    df_mrn[["PatientID", "MRN"]]
    .merge(dob_agg, on="PatientID", how="left")
    .merge(wide_admits, on="PatientID", how="left")
)

# Order columns: PatientID, MRN, DOB, then all InpatientAdmitDate_*
admit_cols = sorted([c for c in final_df.columns if c.startswith("InpatientAdmitDate_")],
                    key=lambda x: int(x.split("_")[-1]))
final_df = final_df[["PatientID", "MRN", "DOB"] + admit_cols]

# ---- Reports ----
no_admit = final_df[admit_cols].isna().all(axis=1).sum() if admit_cols else len(final_df)
print(f"Using sheets -> InpatientAdmitDate: '{name2}', DOB: '{name3}'")
print(f"Patients with NO inpatient admit dates: {no_admit} / {len(final_df)}")
if not dob_conflicts.empty:
    print("\nWARNING: Multiple distinct DOBs found for these PatientIDs (showing counts):")
    print(dob_conflicts.sort_values("PatientID").to_string(index=False))

print("\nFinal table shape:", final_df.shape)
print(final_df.head())

# ---- Save ----
out_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V3/mrn_with_inpatient_dates_dob.xlsx"
final_df.to_excel(out_path, index=False)
print(f"\nSaved merged table to: {out_path}")


Using sheets -> InpatientAdmitDate: 'EncounterAll', DOB: 'PatientInfo'
Patients with NO inpatient admit dates: 2 / 899

Final table shape: (899, 28)
                              PatientID        MRN        DOB  \
0  12288E12-375C-E311-800F-00215A9B0094  038759140 2007-08-25   
1  43C395B6-395C-E311-800F-00215A9B0094  100117962 2010-09-02   
2  D7970BC6-445C-E311-800F-00215A9B0094  100205702 2009-03-16   
3  1912AB08-455C-E311-800F-00215A9B0094  100151965 2000-04-12   
4  1E4D6298-455C-E311-800F-00215A9B0094  041994058 2007-09-11   

  InpatientAdmitDate_1 InpatientAdmitDate_2 InpatientAdmitDate_3  \
0  2024-10-15 21:36:00                  NaT                  NaT   
1  2023-06-18 18:17:00                  NaT                  NaT   
2  2024-01-04 21:17:00                  NaT                  NaT   
3  2024-06-28 11:18:00                  NaT                  NaT   
4  2023-09-13 18:17:00                  NaT                  NaT   

  InpatientAdmitDate_4 InpatientAdmitDate_5 Inpatie

### File Names Regulation

In [30]:
# Add row information to the event list file
import pandas as pd

# Define the file path
event_list_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V3/Raw_Data_6hrsBefore18hrsAfter/1001-1163/Study23_Tag264_EventList.csv"

# Read the CSV file
df = pd.read_csv(event_list_path)

# Create a new column for row numbers
df.insert(0, 'Row Number', [f"Row_{i+1}" for i in range(len(df))])

# Save the modified DataFrame back to the same path
df.to_csv(event_list_path, index=False)

# Display the modified DataFrame
print(df.head())


  Row Number   Mark  Patient ID  Tag ID study_name  study_id  \
0      Row_1  45746        1464     264  PARDS_LWT        23   
1      Row_2  45700        1747     264  PARDS_LWT        23   
2      Row_3  45761        1747     264  PARDS_LWT        23   
3      Row_4  45751        1748     264  PARDS_LWT        23   
4      Row_5  45785        1748     264  PARDS_LWT        23   

                 tag_name          Time Start UTC           Time Stop UTC  \
0  PARDS_Risk_V3_Raw_1163  2025-01-05 08:22:00+00  2025-01-06 08:22:00+00   
1  PARDS_Risk_V3_Raw_1163  2023-12-26 00:49:00+00  2023-12-27 00:49:00+00   
2  PARDS_Risk_V3_Raw_1163  2025-01-06 02:40:00+00  2025-01-07 02:40:00+00   
3  PARDS_Risk_V3_Raw_1163  2023-09-23 17:20:00+00  2023-09-24 17:20:00+00   
4  PARDS_Risk_V3_Raw_1163  2024-02-21 00:48:00+00  2024-02-22 00:48:00+00   

            Time Start            Time Stop  \
0  2025-01-05 03:22:00  2025-01-06 03:22:00   
1  2023-12-25 19:49:00  2023-12-26 19:49:00   
2  2025-01-

In [31]:
import pandas as pd
import os
import glob
import shutil
import re

# Read the event list CSV
event_list_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V3/Raw_Data_6hrsBefore18hrsAfter/1001-1163/Study23_Tag264_EventList.csv"
event_df = pd.read_csv(event_list_path, dtype={"Row Number": str})

# Create a mapping from Row Number to PatientID and StartTime
event_mapping = {str(row["Row Number"]): (row["Patient ID"], row["Time Start"]) for _, row in event_df.iterrows()}

# Define the file paths
directory_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V3/Raw_Data_6hrsBefore18hrsAfter/1001-1163/Study23_Tag264_EventList"

# Ensure the renamed directory exists
renamed_directory = os.path.join(directory_path, "Renamed")
os.makedirs(renamed_directory, exist_ok=True)

# Get all CSV files in the directory
csv_files = glob.glob(os.path.join(directory_path, "*.csv"))

for file_path in csv_files:
    file_name = os.path.basename(file_path)
    
    # Extract the row number from the filename using regex
    match = re.search(r"Row_\d+", file_name)
    if match:
        row_key = match.group()
        
        # Check if extracted row number is in the mapping
        if row_key in event_mapping:
            patient_id, start_time = event_mapping[row_key]
            
            # Convert StartTime to required format
            try:
                formatted_time = pd.to_datetime(start_time).strftime("%Y%m%d_%H")
                cat_name = "V3_6hrsBefore18hrsAfter_Raw"
                new_file_name = f"{patient_id}_{formatted_time}_{cat_name}.csv"
                new_file_path = os.path.join(renamed_directory, new_file_name)
                
                # Copy the file instead of moving it
                shutil.copy2(file_path, new_file_path)
                print(f"Copied and renamed: {file_name} -> {new_file_path}")
            except Exception as e:
                print(f"Error processing file {file_name}: {e}")


Copied and renamed: Event_Row_66_Data_zero_order_interpolation.csv -> /nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V3/Raw_Data_6hrsBefore18hrsAfter/1001-1163/Study23_Tag264_EventList/Renamed/101560_20240901_18_V3_6hrsBefore18hrsAfter_Raw.csv
Copied and renamed: Event_Row_99_Data_zero_order_interpolation.csv -> /nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V3/Raw_Data_6hrsBefore18hrsAfter/1001-1163/Study23_Tag264_EventList/Renamed/340926_20240328_12_V3_6hrsBefore18hrsAfter_Raw.csv
Copied and renamed: Event_Row_152_Data_zero_order_interpolation.csv -> /nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V3/Raw_Data_6hrsBefore18hrsAfter/1001-1163/Study23_Tag264_EventList/Renamed/788076_20250306_11_V3_6hrsBefore18hrsAfter_Raw.csv
Copied and renamed: Event_Row_141_Data_zero_order_interpolation.csv -> /nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V3/Raw_Data_6hrsBefore18hrsAfter/1001-1163/Study23_Tag264_EventList/Renamed/658798_20230912_02_V3

### Create a information table

In [2]:
import pandas as pd
import numpy as np

# Define file path
file_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V3/Raw_Data_6hrsBefore18hrsAfter/user_event_check_tag_280.csv"

# Read the CSV file
df = pd.read_csv(file_path)

# Ensure required columns exist
if "Patient ID" in df.columns and "Time Start" in df.columns:
    # Parse to datetime (keep as datetime; don't .strftime yet)
    ts = pd.to_datetime(df["Time Start"], errors="coerce")

    # Build filenames vectorized:
    # If ts is valid -> PID_YYYYMMDD_HH ; else -> PID_InvalidDate
    pid = df["Patient ID"].astype(str).str.strip()
    ts_fmt = ts.dt.strftime("%Y%m%d_%H")  # yields NaN where ts is NaT

    df["Raw_File_Name"] = np.where(
        ts.notna(),
        pid + "_" + ts_fmt,
        pid + "_InvalidDate"
    )

    # Save the updated DataFrame
    output_path = file_path.replace(".csv", "_updated.csv")
    df.to_csv(output_path, index=False)
    print(f"File saved successfully at: {output_path}")
else:
    print("Error: Required columns 'Patient ID' or 'Time Start' not found in the CSV file.")


File saved successfully at: /nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V3/Raw_Data_6hrsBefore18hrsAfter/user_event_check_tag_280_updated.csv


In [3]:
import pandas as pd

# Define file paths
source_file = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V3/mrn_with_inpatient_dates_dob.xlsx"
target_file = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V3/Raw_Data_6hrsBefore18hrsAfter/user_event_check_tag_280_updated.csv"
output_xlsx = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V3/PARDS_Risk_V3_df.xlsx"

# --- Read the source Excel (correctly) ---
# dtype=str ensures MRN (and others) stay strings; engine=openpyxl is safe for xlsx
df_source = pd.read_excel(source_file, dtype=str, engine="openpyxl")

# --- Read the target CSV with encoding fallbacks ---
try:
    df_target = pd.read_csv(target_file, dtype={"MRN": str}, encoding="utf-8-sig")
except UnicodeDecodeError:
    # Some CSVs are Latin-1/Windows-1252
    df_target = pd.read_csv(target_file, dtype={"MRN": str}, encoding="latin1")

# Ensure 'MRN' column exists in both dataframes
if "MRN" in df_source.columns and "MRN" in df_target.columns:
    # Normalize MRN to 9 digits (keep as strings)
    df_source["MRN"] = df_source["MRN"].str.strip().str.zfill(9)
    df_target["MRN"] = df_target["MRN"].str.strip().str.zfill(9)

    # Convert date-like columns in df_source from the 3rd column onward to a uniform string
    # (If you want to preserve original formatting, remove this block.)
    for col in df_source.columns[2:]:
        df_source[col] = pd.to_datetime(df_source[col], errors="coerce").dt.strftime("%Y-%m-%d %H:%M:%S")

    # Merge on MRN
    df_out = df_target.merge(df_source, on="MRN", how="left")

    # Save to Excel
    df_out.to_excel(output_xlsx, index=False)
    print(f"File updated successfully and saved at: {output_xlsx}")
else:
    print("Error: 'MRN' column not found in one or both files.")


File updated successfully and saved at: /nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V3/PARDS_Risk_V3_df.xlsx


### Identify the starting point of each event.

In [1]:
# Demo
import pandas as pd
import os, re

# Define file paths
xlsx_file = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V3/PARDS_Risk_V3_df.xlsx"
csv_folder = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V3/Raw_Data_6hrsBefore18hrsAfter/Renamed_All"

# Read the Excel file
df_xlsx = pd.read_excel(xlsx_file, dtype={"MRN": str})

# Ventilation type signal groups
vent_sets = {
    "AVEA": ["CAPSULE_AVEAA_VITAL_2564", "CAPSULE_AVEAA_VITAL_1189", "CAPSULE_AVEAA_VITAL_2325", "CAPSULE_AVEAA_WAVE_52911"],
    "CDGR": ["CAPSULE_DRAGERMEDIBUS_VITAL_2564", "CAPSULE_DRAGERMEDIBUS_VITAL_1189", "CAPSULE_DRAGERMEDIBUS_VITAL_2325", "CAPSULE_DRAGERMEDIBUS_WAVE_118"],
    "SVU":  ["CAPSULE_MAQUETC_VITAL_1414", "CAPSULE_MAQUETC_VITAL_1189", "CAPSULE_MAQUETC_VITAL_2325", "MAQUET_CAPSULE_AIRWAY_FLOW_WAVE"],
}

# Helper: extract the base key "PID_YYYYMMDD_HH" from a CSV filename
key_re = re.compile(r"^(\d+_\d{8}_\d{2})")

def csv_to_key(filename_no_ext: str) -> str | None:
    m = key_re.match(filename_no_ext)
    return m.group(1) if m else None

# Get CSV files
csv_files = [f for f in os.listdir(csv_folder) if f.lower().endswith(".csv")]
if not csv_files:
    print("No CSV files found in the folder.")
else:
    # Process the first CSV (or loop over all if you prefer)
    csv_file = csv_files[0]
    file_no_ext = os.path.splitext(csv_file)[0]
    key = csv_to_key(file_no_ext)

    if not key:
        print(f"Could not extract key from filename: {csv_file}")
    else:
        # Find matching row in Excel by Raw_File_Name == key
        matched_row = df_xlsx[df_xlsx["Raw_File_Name"].astype(str).str.strip() == key]

        if matched_row.empty:
            print(f"No match in Excel for key: {key} (from {csv_file})")
        else:
            csv_path = os.path.join(csv_folder, csv_file)
            df_csv = pd.read_csv(csv_path)

            # Identify 1st_Time_Start as first row where one set of signals is non-empty
            if "Time" not in df_csv.columns:
                print(f"'Time' column not found in {csv_file}")
            else:
                df_valid = df_csv.dropna(subset=["Time"])
                first_time_start = None
                vent_type = "Unknown"

                # Iterate rows until we find a vent set fully present & non-null
                for _, row in df_valid.iterrows():
                    for vent, signals in vent_sets.items():
                        # Require columns exist and the row has non-null values for all signals
                        if all(sig in df_valid.columns and pd.notna(row[sig]) for sig in signals):
                            first_time_start = row["Time"]
                            vent_type = vent
                            break
                    if first_time_start is not None:
                        break

                # Results
                print(f"Processing file: {csv_file}")
                print(f"Key (matches Raw_File_Name): {key}")
                print(f"1st_Time_Start: {first_time_start}")
                print(f"Ventilation Type: {vent_type}")


Processing file: 1072814_20240201_20_V3_6hrsBefore18hrsAfter_Raw.csv
Key (matches Raw_File_Name): 1072814_20240201_20
1st_Time_Start: None
Ventilation Type: Unknown


In [2]:
# All — process every CSV and update Excel with 1st_Time_Start + Vent_Type
# Process ALL CSVs in Renamed_All and update Excel with 1st_Time_Start + Ventilation Type
import os, re
import pandas as pd
import numpy as np

# -------- Paths --------
xlsx_file = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V3/PARDS_Risk_V3_df.xlsx"
csv_folder = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V3/Raw_Data_6hrsBefore18hrsAfter/Renamed_All"

# -------- Read Excel --------
df_xlsx = pd.read_excel(xlsx_file, dtype={"MRN": str})
if "Raw_File_Name" not in df_xlsx.columns:
    raise SystemExit("Column 'Raw_File_Name' not found in the Excel file.")

# -------- Vent sets --------
vent_sets = {
    "AVEA": ["CAPSULE_AVEAA_VITAL_2564", "CAPSULE_AVEAA_VITAL_1189",
             "CAPSULE_AVEAA_VITAL_2325", "CAPSULE_AVEAA_WAVE_52911"],
    "CDGR": ["CAPSULE_DRAGERMEDIBUS_VITAL_2564", "CAPSULE_DRAGERMEDIBUS_VITAL_1189",
             "CAPSULE_DRAGERMEDIBUS_VITAL_2325", "CAPSULE_DRAGERMEDIBUS_WAVE_118"],
    "SVU":  ["CAPSULE_MAQUETC_VITAL_1414", "CAPSULE_MAQUETC_VITAL_1189",
             "CAPSULE_MAQUETC_VITAL_2325", "MAQUET_CAPSULE_AIRWAY_FLOW_WAVE"],
}

# -------- Helpers --------
key_re = re.compile(r"^(\d+_\d{8}_\d{2})")  # extracts PID_YYYYMMDD_HH

def csv_to_key(filename_no_ext: str) -> str | None:
    m = key_re.match(filename_no_ext)
    return m.group(1) if m else None

def read_csv_robust(path: str) -> pd.DataFrame:
    try:
        return pd.read_csv(path, encoding="utf-8-sig")
    except UnicodeDecodeError:
        return pd.read_csv(path, encoding="latin1")

def find_first_time_and_vent(df_csv: pd.DataFrame):
    # Require Time column
    if "Time" not in df_csv.columns:
        return None, "Unknown"

    # Keep rows where Time is present (not NaN / not empty string)
    time_ok = df_csv["Time"].notna() & df_csv["Time"].astype(str).str.strip().ne("")
    df_valid = df_csv.loc[time_ok]
    if df_valid.empty:
        return None, "Unknown"

    for vent, signals in vent_sets.items():
        # Ensure all required columns exist
        if not set(signals).issubset(df_valid.columns):
            continue

        sub = df_valid[signals]

        # Build a boolean DataFrame: cell is True if it's non-null and not an empty/whitespace string
        # (We must apply string ops per column, not on the whole DataFrame)
        non_empty_str = sub.apply(lambda s: s.astype(str).str.strip().ne(""))
        mask = sub.notna() & non_empty_str

        # Row qualifies if all required signals are non-empty
        good_rows = mask.all(axis=1)

        if good_rows.any():
            first_idx = good_rows.idxmax()  # first True in original order
            return df_valid.at[first_idx, "Time"], vent

    return None, "Unknown"

# -------- Iterate all CSVs --------
csv_files = sorted([f for f in os.listdir(csv_folder) if f.lower().endswith(".csv")])
if not csv_files:
    print("No CSV files found in the folder.")
else:
    print(f"Found {len(csv_files)} CSV files. Processing...")

first_time_start_map: dict[str, object] = {}
vent_type_map: dict[str, str] = {}
errors = []

for i, csv_file in enumerate(csv_files, 1):
    try:
        base = os.path.splitext(csv_file)[0]
        key = csv_to_key(base)
        if not key:
            print(f"[{i}/{len(csv_files)}] Skip (cannot extract key): {csv_file}")
            continue

        # Only proceed if key exists in Excel Raw_File_Name
        if not (df_xlsx["Raw_File_Name"].astype(str).str.strip() == key).any():
            print(f"[{i}/{len(csv_files)}] Skip (no Excel match): {csv_file} -> key {key}")
            continue

        csv_path = os.path.join(csv_folder, csv_file)
        df_csv = read_csv_robust(csv_path)

        first_time_start, vent_type = find_first_time_and_vent(df_csv)
        first_time_start_map[key] = first_time_start
        vent_type_map[key] = vent_type if first_time_start else "Unknown"

        print(f"[{i}/{len(csv_files)}] OK: {csv_file} -> key={key}, Ventilation Type={vent_type_map[key]}, 1st_Time_Start={first_time_start}")

    except Exception as e:
        errors.append((csv_file, str(e)))
        print(f"[{i}/{len(csv_files)}] ERROR: {csv_file} -> {e}")

# -------- Write back to Excel (add/overwrite columns) --------
df_xlsx["1st_Time_Start"]   = df_xlsx["Raw_File_Name"].astype(str).str.strip().map(first_time_start_map)
df_xlsx["Ventilation Type"] = df_xlsx["Raw_File_Name"].astype(str).str.strip().map(vent_type_map)

# Save (make a backup if you like)
# backup_path = xlsx_file.replace(".xlsx", "_backup.xlsx")
# df_xlsx.to_excel(backup_path, index=False)

df_xlsx.to_excel(xlsx_file, index=False)
print(f"\n✅ Updated {xlsx_file} with '1st_Time_Start' and 'Ventilation Type'.")

if errors:
    print("\n⚠️ Files with errors:")
    for fname, msg in errors[:20]:
        print(f" - {fname}: {msg}")
    if len(errors) > 20:
        print(f" ... and {len(errors) - 20} more")


Found 1089 CSV files. Processing...
[1/1089] OK: 1000780_20250420_12_V3_6hrsBefore18hrsAfter_Raw.csv -> key=1000780_20250420_12, Ventilation Type=CDGR, 1st_Time_Start=2025-04-20 18:08:01.000
[2/1089] OK: 1001941_20231127_17_V3_6hrsBefore18hrsAfter_Raw.csv -> key=1001941_20231127_17, Ventilation Type=Unknown, 1st_Time_Start=None
[3/1089] OK: 1002213_20231128_02_V3_6hrsBefore18hrsAfter_Raw.csv -> key=1002213_20231128_02, Ventilation Type=Unknown, 1st_Time_Start=None
[4/1089] OK: 1009659_20231201_09_V3_6hrsBefore18hrsAfter_Raw.csv -> key=1009659_20231201_09, Ventilation Type=Unknown, 1st_Time_Start=None
[5/1089] OK: 1011066_20231209_13_V3_6hrsBefore18hrsAfter_Raw.csv -> key=1011066_20231209_13, Ventilation Type=Unknown, 1st_Time_Start=None
[6/1089] OK: 1011066_20240521_10_V3_6hrsBefore18hrsAfter_Raw.csv -> key=1011066_20240521_10, Ventilation Type=Unknown, 1st_Time_Start=None
[7/1089] OK: 1011988_20231204_02_V3_6hrsBefore18hrsAfter_Raw.csv -> key=1011988_20231204_02, Ventilation Type=Unkn

Clean the information table

In [ ]:
import pandas as pd
from openpyxl import load_workbook

# -------- Paths --------
xlsx_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V3/PARDS_Risk_V3_df.xlsx"

# -------- Columns to keep --------
keep_cols = ["MRN", "Patient ID", "DOB", "Time Start", "Time Stop", "1st_Time_Start", "Ventilation Type"]
dt_cols   = ["Time Start", "Time Stop", "1st_Time_Start"]  # to format as Excel datetimes

# -------- Read Sheet1 and filter columns --------
df = pd.read_excel(xlsx_path, sheet_name="Sheet1", engine="openpyxl", dtype={"MRN": str})
df = df[[c for c in keep_cols if c in df.columns]]  # keep only existing among requested

# Ensure MRN is 9-digit string
if "MRN" in df.columns:
    df["MRN"] = df["MRN"].fillna("").astype(str).str.strip().str.zfill(9)

# Convert datetime columns to real datetimes (Excel dates)
for c in dt_cols:
    if c in df.columns:
        df[c] = pd.to_datetime(df[c], errors="coerce")  # NaT for unparseable → blank in Excel

# Drop rows where ALL kept columns are empty/NaN
df = df.dropna(how="all", subset=df.columns)

# -------- Write to Sheet2 (replace if exists) --------
with pd.ExcelWriter(xlsx_path, engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
    df.to_excel(writer, sheet_name="Sheet2", index=False)

# -------- Apply Excel number format to datetime columns --------
wb = load_workbook(xlsx_path)
ws = wb["Sheet2"]

# Build header -> column index mapping (1-based)
header_map = {cell.value: cell.column for cell in ws[1] if cell.value is not None}

# Apply date/time display format
date_fmt = "yyyy/mm/dd hh:mm:ss"
for c in dt_cols:
    col_idx = header_map.get(c)
    if col_idx:
        for row in ws.iter_rows(min_row=2, min_col=col_idx, max_col=col_idx):
            cell = row[0]
            # Only set number_format; cells holding NaT remain blank
            cell.number_format = date_fmt

wb.save(xlsx_path)
print(f"✅ Sheet2 created/updated in: {xlsx_path}")


✅ Sheet2 created/updated in: /nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V3/PARDS_Risk_V3_df.xlsx


Rename the raw files

In [ ]:
import pandas as pd
import os
import re
import shutil
from pathlib import Path

# -----------------------------
# Inputs
# -----------------------------
## 1st Raw
# eventlist_csv = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V3/V3_1st_Raw/~V3_1st_Raw_TheOneCausingTrouble~/Study23_Tag292_EventList.csv"
# eventlist_csv = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V3/V3_1st_Raw/~V3_1st_Raw_sub~/Study23_Tag290_EventList.csv"
# eventlist_csv = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V3/V3_1st_Raw/Study23_Tag282_EventList.csv"

# src_dir = Path("/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V3/V3_1st_Raw/~V3_1st_Raw_TheOneCausingTrouble~/Study23_Tag292_EventList")
# src_dir = Path("/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V3/V3_1st_Raw/~V3_1st_Raw_sub~/Study23_Tag290_EventList")
# src_dir = Path("/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V3/V3_1st_Raw/Study23_Tag282_EventList")

# dst_dir = Path("/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V3/V3_1st_Raw/~V3_1st_Raw_TheOneCausingTrouble~/Study23_Tag292_EventList/RENAMED")
# dst_dir = Path("/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V3/V3_1st_Raw/~V3_1st_Raw_sub~/Study23_Tag290_EventList/RENAMED")
# dst_dir = Path("/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V3/V3_1st_Raw/Study23_Tag282_EventList/RENAMED")

## 12th Raw
# eventlist_csv = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V3/V3_12th_Raw/Study23_Tag285_EventList.csv"
# src_dir = Path("/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V3/V3_12th_Raw/Study23_Tag285_EventList")
# dst_dir = Path("/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V3/V3_12th_Raw/Study23_Tag285_EventList/RENAMED")

## 2nd Raw
# eventlist_csv = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V3/V3_2nd_Raw/Study23_Tag283_EventList.csv"
# src_dir = Path("/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V3/V3_2nd_Raw/Study23_Tag283_EventList")
# dst_dir = Path("/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V3/V3_2nd_Raw/Study23_Tag283_EventList/RENAMED")

## 13th Raw
# eventlist_csv = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V3/V3_13th_Raw/~V3_13th_Raw_TheOneCausingTrouble~/Study23_Tag293_EventList.csv"
# eventlist_csv = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V3/V3_13th_Raw/~V3_13th_Raw_sub~/Study23_Tag291_EventList.csv"
# eventlist_csv = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V3/V3_13th_Raw/Study23_Tag286_EventList.csv"

# src_dir = Path("/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V3/V3_13th_Raw/~V3_13th_Raw_TheOneCausingTrouble~/Study23_Tag293_EventList")
# src_dir = Path("/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V3/V3_13th_Raw/~V3_13th_Raw_sub~/Study23_Tag291_EventList")
# src_dir = Path("/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V3/V3_13th_Raw/Study23_Tag286_EventList")

# dst_dir = Path("/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V3/V3_13th_Raw/~V3_13th_Raw_TheOneCausingTrouble~/Study23_Tag293_EventList/RENAMED")
# dst_dir = Path("/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V3/V3_13th_Raw/~V3_13th_Raw_sub~/Study23_Tag291_EventList/RENAMED")
# dst_dir = Path("/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V3/V3_13th_Raw/Study23_Tag286_EventList/RENAMED")

## 3rd Raw
# eventlist_csv = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V3/V3_3rd_Raw/Study23_Tag284_EventList.csv"
# src_dir = Path("/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V3/V3_3rd_Raw/Study23_Tag284_EventList")
# dst_dir = Path("/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V3/V3_3rd_Raw/Study23_Tag284_EventList/RENAMED")

## 14th Raw
eventlist_csv = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V3/V3_14th_Raw/Study23_Tag287_EventList.csv"
src_dir = Path("/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V3/V3_14th_Raw/Study23_Tag287_EventList")
dst_dir = Path("/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V3/V3_14th_Raw/Study23_Tag287_EventList/RENAMED")

# Create destination folder if needed
dst_dir.mkdir(parents=True, exist_ok=True)

# -----------------------------
# Read metadata dataframe
# -----------------------------
df = pd.read_csv(eventlist_csv)

# Ensure required columns exist
required_cols = {"Patient ID", "Time Start"}
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns in EventList CSV: {missing}")

# Parse Time Start
df["Time Start"] = pd.to_datetime(df["Time Start"], errors="coerce")

# -----------------------------
# Helper: extract row index from filename
# Example: Event_Row_1_Data_zero_order_interpolation.csv  -> row_num=1
# -----------------------------
row_pat = re.compile(r"(?:^|_)Row_(\d+)(?:_|\.|$)", re.IGNORECASE)

def extract_row_number(filename: str) -> int:
    m = row_pat.search(filename)
    if not m:
        raise ValueError(f"Could not extract row number from filename: {filename}")
    return int(m.group(1))

# -----------------------------
# Copy + rename
# -----------------------------
# We'll iterate all files in src_dir (adjust filter if needed)
src_files = sorted([p for p in src_dir.iterdir() if p.is_file()])

# Optional: restrict to CSVs only
# src_files = [p for p in src_files if p.suffix.lower() == ".csv"]

# Track potential collisions
used_names = set()

for p in src_files:
    try:
        row_num = extract_row_number(p.name)
    except ValueError as e:
        print(f"SKIP (no row found): {p.name}")
        continue

    # IMPORTANT: decide whether Row_1 corresponds to df index 0 or 1.
    # Most "Row_1" naming means first row -> df index 0, so subtract 1:
    df_idx = row_num - 1

    if df_idx < 0 or df_idx >= len(df):
        print(f"SKIP (row out of range): {p.name} -> Row_{row_num} maps to df[{df_idx}] but df has {len(df)} rows")
        continue

    patient_id = df.loc[df_idx, "Patient ID"]
    ts = df.loc[df_idx, "Time Start"]

    if pd.isna(ts):
        print(f"SKIP (bad Time Start): {p.name} -> df[{df_idx}] Time Start is NaT")
        continue

    ts_str = ts.strftime("%Y%m%d_%H")
    # new_name = f"{patient_id}_{ts_str}_1st_Raw.csv"
    # new_name = f"{patient_id}_{ts_str}_12th_Raw.csv"
    # new_name = f"{patient_id}_{ts_str}_2nd_Raw.csv"
    # new_name = f"{patient_id}_{ts_str}_13th_Raw.csv"
    # new_name = f"{patient_id}_{ts_str}_3rd_Raw.csv"
    new_name = f"{patient_id}_{ts_str}_14th_Raw.csv"    

    # Collision protection (in case two rows create the same name)
    final_name = new_name
    k = 2
    while final_name in used_names or (dst_dir / final_name).exists():
        # final_name = new_name.replace("_1st_Raw.csv", f"_1st_Raw_v{k}.csv")
        # final_name = new_name.replace("_12th_Raw.csv", f"_12th_Raw_v{k}.csv")
        # final_name = new_name.replace("_2nd_Raw.csv", f"_2nd_Raw_v{k}.csv")
        # final_name = new_name.replace("_13th_Raw.csv", f"_13th_Raw_v{k}.csv")
        # final_name = new_name.replace("_3rd_Raw.csv", f"_3rd_Raw_v{k}.csv")
        final_name = new_name.replace("_14th_Raw.csv", f"_14th_Raw_v{k}.csv")
        k += 1

    used_names.add(final_name)

    src_path = p
    dst_path = dst_dir / final_name

    shutil.copy2(src_path, dst_path)  # copy2 preserves timestamps/metadata
    print(f"COPIED: {src_path.name} -> {dst_path.name}")

print(f"\nDone. Renamed copies are in:\n{dst_dir}")


Calculate the Age (Should be done early in Sheet1, but forgot......)

In [2]:
import pandas as pd

excel_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V3/PARDS_Risk_V3_df.xlsx"

# --- Read sheets ---
df_s1 = pd.read_excel(excel_path, sheet_name="Sheet1")
df_s6 = pd.read_excel(excel_path, sheet_name="Sheet6")

# --- Check required columns ---
for col in ["Patient ID", "DOB"]:
    if col not in df_s1.columns:
        raise ValueError(f"Sheet1 missing column: {col}")

for col in ["Patient ID", "1st_Time_Start"]:
    if col not in df_s6.columns:
        raise ValueError(f"Sheet6 missing column: {col}")

# --- Parse dates ---
df_s1["DOB"] = pd.to_datetime(df_s1["DOB"], errors="coerce")
df_s6["1st_Time_Start"] = pd.to_datetime(df_s6["1st_Time_Start"], errors="coerce")

# --- Build a 1:1 Patient ID -> DOB mapping (prevents row duplication) ---
# If a patient has multiple DOB entries in Sheet1, this keeps the first non-null DOB.
dob_map = (
    df_s1[["Patient ID", "DOB"]]
    .dropna(subset=["Patient ID"])
    .sort_values("DOB")
    .drop_duplicates(subset=["Patient ID"], keep="first")
    .set_index("Patient ID")["DOB"]
)

# --- Add DOB to Sheet6 without changing row count ---
df_out = df_s6.copy()
df_out["DOB"] = df_out["Patient ID"].map(dob_map)

# --- Compute Age in years (2 decimals) ---
df_out["Age_Years"] = ((df_out["1st_Time_Start"] - df_out["DOB"]) / pd.Timedelta(days=365.25)).round(2)

# --- Save back to Sheet6 (replace Sheet6 only) ---
with pd.ExcelWriter(excel_path, engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
    df_out.to_excel(writer, sheet_name="Sheet6", index=False)

print(f"✅ Done. Sheet6 rows preserved: {len(df_s6)} -> {len(df_out)}")


✅ Done. Sheet6 rows preserved: 194 -> 194


In [1]:
from openpyxl import load_workbook
import pandas as pd

file_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V3/PARDS_Risk_V3_df.xlsx"

# ------------------------------------------------------------
# Read Sheet2
# ------------------------------------------------------------
df = pd.read_excel(file_path, sheet_name="Sheet2")

# ------------------------------------------------------------
# Parse 1st_Time_Start as datetime
# ------------------------------------------------------------
df["1st_Time_Start"] = pd.to_datetime(df["1st_Time_Start"], errors="coerce")

# ------------------------------------------------------------
# 1) Create additional Time_Start columns FIRST
#    Each is 1st_Time_Start + N hours
# ------------------------------------------------------------
hour_offsets = {
    "2nd": 1,
    "3rd": 2,
    "12th": 12,
    "13th": 13,
    "14th": 14,
}

for ordinal, hrs in hour_offsets.items():
    df[f"{ordinal}_Time_Start"] = df["1st_Time_Start"] + pd.to_timedelta(hrs, unit="h")

# ------------------------------------------------------------
# 2) Create relative file names:
#    PatientID_YYYYMMDD_HH_<ordinal>
# ------------------------------------------------------------
def make_fname(patient_id, ts, ordinal):
    if pd.isna(patient_id) or pd.isna(ts):
        return pd.NA
    ts = pd.Timestamp(ts)
    return f"{int(patient_id)}_{ts.strftime('%Y%m%d')}_{ts.strftime('%H')}_{ordinal}"

all_ordinals = ["1st"] + list(hour_offsets.keys())

# 1st uses the original time
df["1st_rel_fname"] = [
    make_fname(pid, ts, "1st")
    for pid, ts in zip(df["Patient ID"], df["1st_Time_Start"])
]

# Others use the offset times
for o in hour_offsets:
    df[f"{o}_rel_fname"] = [
        make_fname(pid, ts, o)
        for pid, ts in zip(df["Patient ID"], df[f"{o}_Time_Start"])
    ]

# ------------------------------------------------------------
# 3) FILTER: keep only rows with non-empty 1st_Time_Start
# ------------------------------------------------------------
df_filtered = df[df["1st_Time_Start"].notna()].copy()

# ------------------------------------------------------------
# Write to Sheet3 (replace Sheet3 if it already exists)
# ------------------------------------------------------------
with pd.ExcelWriter(
    file_path,
    engine="openpyxl",
    mode="a",
    if_sheet_exists="replace"
) as writer:
    df_filtered.to_excel(writer, sheet_name="Sheet3", index=False)
